# CXR-LLaVA Colab T4 smoke test

Research-only notebook for loading CXR-LLaVA v2 with 4-bit NF4 Llama weights and generating one report from a sample chest X-ray. Enable a Google Colab GPU runtime and select a T4 when available.

When switching from the previous full-precision model: run the updated pip cell once, restart the session to release the old model and failed generation state, then rerun configuration/clone and the remaining cells (skip pip after restarting). Downloaded checkpoint files are reused from the Hugging Face cache when the runtime disk is retained.

> Model output is experimental and must not be used for clinical diagnosis, treatment, or patient-specific medical decisions.

## 1. Configure the fork

Set `GITHUB_REPO_URL` to your public fork. Leave it empty only when the notebook is opened from an already cloned checkout. Do not put access tokens in this notebook.

In [ ]:
import os
import subprocess
from pathlib import Path

GITHUB_REPO_URL = "https://github.com/hanhpm/CXR_LLaVA_Improvement.git"
GITHUB_BRANCH = "master"
PROJECT_DIR = Path("/content/CXR_LLaVA_Improvement")
MODEL_ID = "ECOFRI/CXR-LLAVA-v2"
SAMPLE_IMAGE = PROJECT_DIR / "IMG" / "img.jpg"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", "--branch", GITHUB_BRANCH, GITHUB_REPO_URL, str(PROJECT_DIR)], check=True)
else:
    print("Using existing checkout:", PROJECT_DIR)

os.chdir(PROJECT_DIR)
print("Working directory:", Path.cwd())

In [ ]:
%pip uninstall -y transformers tokenizers
%pip install -q \
    "transformers==4.46.3" \
    "tokenizers<0.21" \
    "huggingface-hub==0.36.0" \
    "protobuf==5.29.6" \
    sentencepiece \
    accelerate \
    "bitsandbytes>=0.43.3" \
    pillow

In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    print("Warning: no CUDA GPU detected; model loading may exceed CPU memory and is not recommended.")

In [ ]:
from IPython.display import display
from PIL import Image

if not SAMPLE_IMAGE.exists():
    raise FileNotFoundError(f"Sample image not found: {SAMPLE_IMAGE}")
sample_image = Image.open(SAMPLE_IMAGE)
print("Image:", SAMPLE_IMAGE)
print("Original mode and size:", sample_image.mode, sample_image.size)
display(sample_image.convert("L"))

## 2. Load the model in 4-bit

Llama Linear layers use NF4 with double quantization and float16 compute. The vision tower, multimodal projector and output head are excluded from quantization to preserve the custom vision code, direct attention weight access and output head precision. Other floating-point tensors retain the dtypes selected by the loader/custom model; this is not a global FP16 conversion.

Quantization happens while loading the original checkpoint; it reduces GPU weight memory, not the original download size. This is a 4-bit smoke test and its outputs may differ from the unquantized baseline. Configuration reference: [Transformers bitsandbytes](https://huggingface.co/docs/transformers/v4.46.3/en/quantization/bitsandbytes).

Use an explicit device map (`{"": 0}`) to place the entire model on GPU 0. The custom model does not support automatic splitting (`device_map="auto"`). This requires enough free GPU memory for weights and inference; it does not offload to CPU. Do not call `model.to(device)` after loading. CUDA is required because the report helper calls `.cuda()` internally.

In [ ]:
import torch
import transformers
import bitsandbytes as bnb
from transformers import AutoModel, BitsAndBytesConfig
from unittest.mock import patch

if not torch.cuda.is_available():
    raise RuntimeError("Enable a GPU runtime in Colab before loading CXR-LLaVA.")
if "model" in globals():
    raise RuntimeError("A model already exists. Reuse it or restart the session before loading another copy.")
torch.cuda.set_device(0)

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
    # Transformers 4.46.3 also uses this skip list for 4-bit loading.
    llm_int8_skip_modules=["vision_tower", "mm_projector", "lm_head"],
)

original_from_pretrained = transformers.LlamaTokenizer.from_pretrained

def patched_from_pretrained(*args, **kwargs):
    kwargs.pop("add_special_tokens", None)
    return original_from_pretrained(*args, **kwargs)

with patch.object(
    transformers.LlamaTokenizer,
    "from_pretrained",
    new=patched_from_pretrained,
):
    model = AutoModel.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        torch_dtype=torch.float16,
        quantization_config=quantization_config,
        low_cpu_mem_usage=True,
        device_map={"": 0},
    )
model.eval()
quantized_layer_count = sum(isinstance(layer, bnb.nn.Linear4bit) for layer in model.llama.modules())
assert getattr(model, "is_loaded_in_4bit", False) and quantized_layer_count > 0, "4-bit loading was not applied."
assert not any(isinstance(layer, bnb.nn.Linear4bit) for layer in model.vision_tower.modules()), "Vision tower was unexpectedly quantized."
print("Loaded:", MODEL_ID)
print("Device map:", model.hf_device_map)
print("Llama 4-bit Linear layers:", quantized_layer_count)
print("PyTorch allocated GiB:", round(torch.cuda.memory_allocated(0) / 1024**3, 2))
print("Free GPU GiB:", round(torch.cuda.mem_get_info(0)[0] / 1024**3, 2))

## 3. Restore the chat template

Run this cell after loading and before report/Q&A inference. It also works on the model already in RAM after the missing-template error; no restart or checkpoint reload is needed. Existing templates are preserved. The fallback supports the explicit system message supplied by both CXR-LLaVA helpers; custom chats should also supply their system message.

Format reference: [Transformers v4.36.2 LlamaTokenizer](https://github.com/huggingface/transformers/blob/v4.36.2/src/transformers/models/llama/tokenization_llama.py). Passing the formatting check does not yet confirm GPU generation.

In [ ]:
# Restore the legacy Llama 2 format used by CXR-LLaVA's report/Q&A helpers.
# Reference: Transformers v4.36.2, LlamaTokenizer.default_chat_template (Apache-2.0).
if not model.tokenizer.chat_template:
    model.tokenizer.chat_template = (
        "{% if messages[0]['role'] == 'system' %}"
        "{% set system_message = messages[0]['content'] %}"
        "{% set turns = messages[1:] %}"
        "{% else %}"
        "{% set system_message = false %}"
        "{% set turns = messages %}"
        "{% endif %}"
        "{% for message in turns %}"
        "{% if (message['role'] == 'user') != (loop.index0 % 2 == 0) %}"
        "{{ raise_exception('Roles must alternate user/assistant.') }}"
        "{% endif %}"
        "{% set content = message['content'] %}"
        "{% if loop.index0 == 0 and system_message != false %}"
        "{% set content = '<<SYS>>\\n' + system_message + '\\n<</SYS>>\\n\\n' + content %}"
        "{% endif %}"
        "{% if message['role'] == 'user' %}"
        "{{ bos_token + '[INST] ' + content.strip() + ' [/INST]' }}"
        "{% elif message['role'] == 'assistant' %}"
        "{{ ' ' + content.strip() + ' ' + eos_token }}"
        "{% endif %}"
        "{% endfor %}"
    )

# Validate formatting without running the GPU model.
template_check_chat = [
    {"role": "system", "content": "You are a helpful radiologist."},
    {"role": "user", "content": "<image>\nWrite a radiologic report."},
]
template_check_prompt = model.apply_chat_template(template_check_chat)
assert template_check_prompt == (
    model.tokenizer.bos_token
    + "[INST] <<SYS>>\nYou are a helpful radiologist.\n<</SYS>>\n\n"
    + "<image>\nWrite a radiologic report. [/INST]"
)
print("Chat template ready; prompt formatting check passed.")


In [ ]:
# Single-image smoke test using the documented repository API.
with torch.inference_mode():
    report = model.write_radiologic_report(sample_image)

print("MODEL-GENERATED REPORT (4-bit NF4, research only):")
print(report)

In [ ]:
# Optional question-answering smoke test.
question = "What findings should be reviewed by a qualified radiologist?"
with torch.inference_mode():
    answer = model.ask_question(question=question, image=sample_image)
print("QUESTION:", question)
print("MODEL-GENERATED ANSWER (research only):")
print(answer)

## Result classification

If the image loads and the two calls return text, this notebook has passed a **single-image 4-bit NF4 inference smoke test**. Record quantization when comparing results with the unquantized baseline. It is not a benchmark, clinical validation, or metric-complete evaluation. The CSV is intentionally not processed by default.

In [ ]:
# Optional cleanup before another model run.
import gc

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print("Model memory released.")